# Q_3: Pick Your Own Dataset


## 1. Find a dataset on a topic you're interested in. Some easy options are data.gov, kaggle.com, and data.world.


I picked a PGA Tour dataset that I created for a project last semester. Our dependent variable will be money_earned (success on the tour).

## 2. Clean the data and do some exploratory data analysis on key variables that interest you. Pick a particular target/outcome variable and features/predictors.


In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [28]:
data = pd.read_csv("PGA.csv")
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2515 entries, 0 to 2514
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   year                  2515 non-null   int64  
 1   player_name           2515 non-null   str    
 2   sg_total              1895 non-null   float64
 3   sg_off_the_tee        1895 non-null   float64
 4   sg_approach           1895 non-null   float64
 5   sg_around_green       1895 non-null   float64
 6   sg_putting            1895 non-null   float64
 7   driving_distance      1895 non-null   float64
 8   driving_accuracy      1895 non-null   float64
 9   greens_in_regulation  1895 non-null   float64
 10  scoring_average       1892 non-null   float64
 11  money_earned          2503 non-null   float64
 12  final_season_rank     1766 non-null   float64
 13  sg_tee_to_green       1895 non-null   float64
dtypes: float64(12), int64(1), str(1)
memory usage: 275.2 KB


In [29]:
# drop player name because it is irrelevant to this question.
# also final season rank is another success metric but I don't want to look at that 
# I will remove it because I don't think it will be helpful
# too directly corelated to success
# also dropping year because the is irrelevant to this calculation
keep = ['sg_total', 'sg_off_the_tee', 'sg_approach',
       'sg_around_green', 'sg_putting', 'driving_distance', 
       'driving_accuracy', 'greens_in_regulation', 'scoring_average', 
       'money_earned','sg_tee_to_green'
       ]

data = data[keep]

In [30]:
# there is a large skew in the money_earned category
# lets take the log of it

data["log_money_earned"] = np.log(data["money_earned"])

In [31]:
# check missing values in each category
data.isnull().sum(axis=0)

sg_total                620
sg_off_the_tee          620
sg_approach             620
sg_around_green         620
sg_putting              620
driving_distance        620
driving_accuracy        620
greens_in_regulation    620
scoring_average         623
money_earned             12
sg_tee_to_green         620
log_money_earned         12
dtype: int64

In [32]:
# drop rows with missing values

data.dropna(inplace=True)

In [33]:
data.isnull().sum(axis=0)

sg_total                0
sg_off_the_tee          0
sg_approach             0
sg_around_green         0
sg_putting              0
driving_distance        0
driving_accuracy        0
greens_in_regulation    0
scoring_average         0
money_earned            0
sg_tee_to_green         0
log_money_earned        0
dtype: int64

In [34]:
golf = data.copy()

## 3. Split the sample into an ~80% training set and a ~20% test set.


In [35]:
golf.columns

Index(['sg_total', 'sg_off_the_tee', 'sg_approach', 'sg_around_green',
       'sg_putting', 'driving_distance', 'driving_accuracy',
       'greens_in_regulation', 'scoring_average', 'money_earned',
       'sg_tee_to_green', 'log_money_earned'],
      dtype='str')

In [36]:
cols = ['sg_total', 'sg_off_the_tee', 'sg_approach', 'sg_around_green',
       'sg_putting', 'driving_distance', 'driving_accuracy',
       'greens_in_regulation', 'scoring_average', 'sg_tee_to_green'
       ]

X_final = golf[cols]
y_final = golf.loc[X_final.index, 'log_money_earned']

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final, test_size=0.2, random_state=42)

## 4. Run a few regressions of your target/outcome variable on a variety of features/predictors. Compute the RMSE on the test set.

In [38]:
# first model will be with the simple metrics that don't compare yourself to the field directly

cols = ['driving_distance', 'driving_accuracy',
       'greens_in_regulation', 'scoring_average'
       ]

model_numeric = LinearRegression(fit_intercept=True).fit(X_train[cols], y_train)

y_pred = model_numeric.predict(X_test[cols])

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² Score: {r2:.4f}")

Mean Squared Error (MSE): 0.33
Root Mean Squared Error (RMSE): 0.57
R² Score: 0.7034


In [39]:
# second model will be with advacned metrics that compare a player directly to the field

cols = ['sg_total', 'sg_off_the_tee', 'sg_approach', 'sg_around_green',
       'sg_putting','sg_tee_to_green'
       ]

model_numeric = LinearRegression(fit_intercept=True).fit(X_train[cols], y_train)

y_pred = model_numeric.predict(X_test[cols])

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² Score: {r2:.4f}")

Mean Squared Error (MSE): 0.40
Root Mean Squared Error (RMSE): 0.64
R² Score: 0.6354


In [40]:
# now I am going to combine the two together

cols = ['sg_total', 'sg_off_the_tee', 'sg_approach', 'sg_around_green',
       'sg_putting', 'driving_distance', 'driving_accuracy',
       'greens_in_regulation', 'scoring_average', 'sg_tee_to_green'
       ]


model_numeric = LinearRegression(fit_intercept=True).fit(X_train[cols], y_train)

y_pred = model_numeric.predict(X_test[cols])

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² Score: {r2:.4f}")

Mean Squared Error (MSE): 0.32
Root Mean Squared Error (RMSE): 0.56
R² Score: 0.7146


## 5. Which model performed the best, and why?


The third model that took into account both types of analytical data did better than the other two models that only took into account one type of data. This is because all of the columns were to some extent relevant to the performance of the golfer. If you took some of them away, the explainability would not be as high, and the error would be higher. However, if you put them together, you are taking into account different facets of the golfer's game that might help you understand how well they are performing.

## 6. What did you learn?

I learned that many different types of data can be used to explain an outcome. Sometimes, all that matters is how you manipulate the information. It is important to have a combination of both categorical and numerical data if possible. To that point, the more diverse your data is, the better it might be. However, it is important that the data you are using is relevant. If data is not relevant to the question, it might not help you create a stronger model. A great example of this would be "player_name. In some cases knowing the name of each player would be important, but for the sake of understanding the dependent variable "money_earned", knowing the player name would not be very relevant. We aren't grouping the output by player name.